# Create YouTube Title-Thumbnail Training Pairs

Code authored by: Shaw Talebi

[Video link](https://youtu.be/W4s6b2ZM6kI) | [Blog link](https://medium.com/towards-data-science/fine-tuning-multimodal-embedding-models-bf007b1c5da5) <br>
[Dataset](https://huggingface.co/datasets/shawhin/yt-title-thumbnail-pairs) | [Fine-tuned Model](https://huggingface.co/shawhin/clip-title-thumbnail-embeddings)

### install dependencies

In [ ]:
%%bash

which python

pip install -r requirements.txt

### imports

In [1]:
from top_secret import my_key
import requests
from isodate import parse_duration

import pandas as pd
import numpy as np
# https://huggingface.co/sentence-transformers/all-mpnet-base-v2
from sentence_transformers import SentenceTransformer
from datasets import DatasetDict, Dataset

### Extract

#### extract video ids

In [3]:
channel_id = 'UCa9gErQ9AE5jT2DZLjXBIdA' # my YouTube channel ID
page_token = None # initialize page token
url = 'https://www.googleapis.com/youtube/v3/search' # YouTube search API endpoint

# extract video data across multiple search result pages
video_id_list = []

# print(my_key)

while page_token != 0:
    params = {
        "key": my_key, 
        'channelId': channel_id, 
        'part': ["snippet","id"], 
        'order': "date", 
        'maxResults':50, 
        'pageToken': page_token
    }
    response = requests.get(url, params=params)

    # print(dict(response.json()))

    for raw_item in dict(response.json())['items']:
        
        # only execute for youtube videos
        if raw_item['id']['kind'] != "youtube#video":
            continue

        # grab video ids
        video_id_list.append(raw_item['id']['videoId'])

    try:
        # grab next page token
        page_token = dict(response.json())['nextPageToken']
    except:
        # if no next page token kill while loop
        page_token = 0

In [4]:
len(video_id_list)

153

#### extract titles and thumbnail urls

In [5]:
url = "https://www.googleapis.com/youtube/v3/videos"

video_data_list = []

for video_id in video_id_list:

    params = {
        "part": ["snippet","contentDetails"],
        "id": video_id,  
        "key": my_key,  
    }
    response = requests.get(url, params=params)
    
    raw_dict = dict(response.json())['items'][0]

    # only process videos longer than 3 minutes
    iso_duration = raw_dict['contentDetails']["duration"]
    if parse_duration(iso_duration).total_seconds() < 180:
        continue
    
    # extract video data
    video_data = {}
    video_data['video_id'] = video_id
    video_data['title'] = raw_dict['snippet']['title']
    video_data['thumbnail_url'] = raw_dict['snippet']['thumbnails']['high']['url']

    # append data to list
    video_data_list.append(video_data)

In [6]:
len(video_data_list)

108

### Transform

#### create dataframe

In [7]:
df = pd.DataFrame(video_data_list)
df.head()

,video_id,title,thumbnail_url
0,PWpNWGBKr5A,7 Skills I Had to Learn to Make $100k/yr (as a...,https://i.ytimg.com/vi/PWpNWGBKr5A/hqdefault.jpg
1,f7mzI0NVjP8,Exposing My AI SaaS Tech Stack (so you can cop...,https://i.ytimg.com/vi/f7mzI0NVjP8/hqdefault.jpg
2,QxLXhE1fxc4,uv: The Fastest Way to Install (and Use) Python,https://i.ytimg.com/vi/QxLXhE1fxc4/hqdefault.jpg
3,enBm0jLXLZ4,GitHub for AI Engineers (beginner-friendly guide),https://i.ytimg.com/vi/enBm0jLXLZ4/hqdefault.jpg
4,zKHSpwayPBU,Context Engineering Explained (5 Practical Tips),https://i.ytimg.com/vi/zKHSpwayPBU/hqdefault.jpg


#### create negative pairs

In [8]:
# Load the model
model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
%%time
# Encode all titles
job_embeddings = model.encode(df['title'].to_list())
print(job_embeddings.shape)

(108, 768)
CPU times: user 623 ms, sys: 521 ms, total: 1.14 s
Wall time: 6.29 s


In [10]:
# compute similarities
similarities = model.similarity(job_embeddings, job_embeddings)
print(similarities.shape)

torch.Size([108, 108])


In [12]:
# match least title least similar to positive match as the negative match
similarities_argsorted = np.argsort(similarities.numpy(), axis=1)
negative_pair_index_list = []

for i in range(len(similarities)):

    # Start with the smallest similarity index for the current row
    j = 0
    index = int(similarities_argsorted[i][j])

    # Ensure the index is unique
    while index in negative_pair_index_list:
        j += 1  # Move to the next smallest index
        index = int(similarities_argsorted[i][j])  # Fetch next smallest index

    negative_pair_index_list.append(index)

In [17]:
# add negative pairs to df
df['title_neg'] = df['title'].iloc[negative_pair_index_list].values

In [18]:
df.head()

,video_id,title,thumbnail_url,title_neg
0,PWpNWGBKr5A,7 Skills I Had to Learn to Make $100k/yr (as a...,https://i.ytimg.com/vi/PWpNWGBKr5A/hqdefault.jpg,Persistent Homology | Introduction & Python Ex...
1,f7mzI0NVjP8,Exposing My AI SaaS Tech Stack (so you can cop...,https://i.ytimg.com/vi/f7mzI0NVjP8/hqdefault.jpg,"Pareto, Power Laws, and Fat Tails"
2,QxLXhE1fxc4,uv: The Fastest Way to Install (and Use) Python,https://i.ytimg.com/vi/QxLXhE1fxc4/hqdefault.jpg,Why Conflict Is Good & How You Can Use It
3,enBm0jLXLZ4,GitHub for AI Engineers (beginner-friendly guide),https://i.ytimg.com/vi/enBm0jLXLZ4/hqdefault.jpg,What Nature Can Teach Us About Business...
4,zKHSpwayPBU,Context Engineering Explained (5 Practical Tips),https://i.ytimg.com/vi/zKHSpwayPBU/hqdefault.jpg,"Why I Quit My $150,000 Data Science Job"


#### train-test split

In [19]:
# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into train, validation, and test sets (e.g., 80% train, 20% test)
train_frac = 0.7
valid_frac = 0.15
test_frac = 0.15

# define train and validation size
train_size = int(train_frac * len(df))
valid_size = int(valid_frac * len(df))

# create train, validation, and test datasets
df_train = df[:train_size]
df_valid = df[train_size:train_size + valid_size]
df_test = df[train_size + valid_size:]

### Load

In [20]:
# Convert the pandas DataFrames back to Hugging Face Datasets
train_ds = Dataset.from_pandas(df_train)
valid_ds = Dataset.from_pandas(df_valid)
test_ds = Dataset.from_pandas(df_test)

# Combine into a DatasetDict
dataset_dict = DatasetDict({
    'train': train_ds,
    'valid': valid_ds,
    'test': test_ds
})

In [21]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['video_id', 'title', 'thumbnail_url', 'title_neg'],
        num_rows: 75
    })
    valid: Dataset({
        features: ['video_id', 'title', 'thumbnail_url', 'title_neg'],
        num_rows: 16
    })
    test: Dataset({
        features: ['video_id', 'title', 'thumbnail_url', 'title_neg'],
        num_rows: 17
    })
})

In [24]:
# push data to hub
dataset_dict.push_to_hub("pkqinys/shawhin-yt-title-thumbnail-pairs")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/pkqinys/shawhin-yt-title-thumbnail-pairs/commit/24578b7f6e0a3aa31e829ad3580ef0427dde8c36', commit_message='Upload dataset', commit_description='', oid='24578b7f6e0a3aa31e829ad3580ef0427dde8c36', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pkqinys/shawhin-yt-title-thumbnail-pairs', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pkqinys/shawhin-yt-title-thumbnail-pairs'), pr_revision=None, pr_num=None)